In [1]:
import sympy as sp
import numpy as np
from IPython.display import display, Math

In [2]:
# function definitions - proceed to the next cell for calculations

def compute_christoffel_universal(metric_sp, coords):
    """
    Computes exact symbolic \\Gamma^\\rho_{\\mu\\nu} for ANY coordinate system. (Used \\ to prevent escape sequences)
    """
    # We dynamically build the \nabla vector based on the provided coordinates.
    # We use 'c=c' in the lambda to bind the current coordinate in the loop,
    # preventing Python's late-binding closure issue.
    nabla_vec = [
        lambda T, c=c: np.vectorize(lambda expr: sp.diff(expr, c))(T)
        for c in coords
    ]
    
    # 2. Convert SymPy to NumPy object arrays
    g_inv_sp = metric_sp.inv()
    g = np.array(metric_sp.tolist(), dtype=object)
    g_inv = np.array(g_inv_sp.tolist(), dtype=object)
    
    # 3. Apply \nabla to the metric
    dg = np.stack([op(g) for op in nabla_vec], axis=0)
    
    # 4. Einstein Summation for the contractions
    term1 = np.einsum('kl, ijl -> kij', g_inv, dg)  
    term2 = np.einsum('kl, jil -> kij', g_inv, dg)  
    term3 = np.einsum('kl, lij -> kij', g_inv, dg)  
    
    Gamma_unsimplified = (term1 + term2 - term3)/2
    
    # 5. Simplify and return
    simplify_array = np.vectorize(sp.simplify)
    return simplify_array(Gamma_unsimplified)

def compute_riemann(Gamma, coords):
    """
    Computes exact symbolic Riemann curvature tensor R^\\rho_{\\sigma\\mu\\nu}.
    Gamma should be the 3D NumPy array of Christoffel symbols.
    """
    # 1. Dynamically build the \nabla vector based on the provided coordinates.
    nabla_vec = [
        lambda T, c=c: np.vectorize(lambda expr: sp.diff(expr, c))(T)
        for c in coords
    ]
    
    # 2. Apply \nabla to \Gamma.
    # dG has shape (dim, dim, dim, dim).
    # dG[i, rho, mu, nu] represents \partial_i \Gamma^\rho_{\mu\nu}
    dG = np.stack([op(Gamma) for op in nabla_vec], axis=0)

    # 3. Einstein Summation for the four terms of the Riemann tensor
    # R^\rho_{\sigma\mu\nu} (Indices: r=rho, s=sigma, m=mu, n=nu, l=lambda)
    
    # \partial_\mu \Gamma^\rho_{\nu\sigma}
    term1 = np.einsum('mrns -> rsmn', dG)
    
    # \partial_\nu \Gamma^\rho_{\mu\sigma}
    term2 = np.einsum('nrms -> rsmn', dG)
    
    # \Gamma^\rho_{\mu\lambda} \Gamma^\lambda_{\nu\sigma}
    term3 = np.einsum('rml, lns -> rsmn', Gamma, Gamma)
    
    # \Gamma^\rho_{\nu\lambda} \Gamma^\lambda_{\mu\sigma}
    term4 = np.einsum('rnl, lms -> rsmn', Gamma, Gamma)

    # Combine terms
    R_unsimplified = term1 - term2 + term3 - term4

    # 4. Simplify and return
    simplify_array = np.vectorize(sp.simplify)
    return simplify_array(R_unsimplified)

def compute_ricci_tensor(R_tensor):
    """
    Computes exact symbolic Ricci tensor R_{\\mu\\nu}.
    Contracts the Riemann tensor R^\\rho_{\\sigma\\mu\\nu} over rho and mu.
    """
    # R_tensor has axes (rho, sigma, mu, nu)
    # We want to sum over rho (axis 0) and mu (axis 2).
    # 'ijil -> jl' contracts index 'i', leaving 'j' (sigma) and 'l' (nu)
    Ricci_unsimplified = np.einsum('ijil -> jl', R_tensor)
    
    simplify_array = np.vectorize(sp.simplify)
    return simplify_array(Ricci_unsimplified)

def compute_ricci_scalar(Ricci_tensor, metric_sp):
    """
    Computes the Ricci scalar R = g^{\\mu\\nu} R_{\\mu\\nu}
    """
    g_inv_sp = metric_sp.inv()
    g_inv = np.array(g_inv_sp.tolist(), dtype=object)
    
    # Contract the inverse metric with the Ricci tensor
    R_scalar_unsimplified = np.einsum('ij, ij ->', g_inv, Ricci_tensor)
    
    return sp.simplify(R_scalar_unsimplified)

def compute_einstein_tensor(Ricci_tensor, Ricci_scalar, metric_sp):
    
    g_inv_sp = metric_sp.inv()
    g = np.array(metric_sp.tolist(), dtype=object)
    g_inv = np.array(g_inv_sp.tolist(), dtype=object)

    G_unsimplified = Ricci_tensor - (Ricci_scalar * g)/2

    return sp.simplify(G_unsimplified)

def print_non_zero_symbols(Gamma_tensor, coord_names):
    """Helper function to print only non-zero \\Gamma symbols."""
    dim = len(coord_names)
    for rho in range(dim):
        for mu in range(dim):
            for nu in range(dim):
                val = Gamma_tensor[rho, mu, nu]
                if val != 0:
                    val_latex = sp.latex(val)
                    eq_str = rf"\Gamma^{coord_names[rho]}_{{{coord_names[mu]} \,{coord_names[nu]}}} = {val_latex}"
                    display(Math(eq_str))

def print_riemann_symbols(R_tensor, coord_names):
    """Helper function to print only non-zero Riemann components."""
    dim = len(coord_names)
    for rho in range(dim):
        for sigma in range(dim):
            for mu in range(dim):
                for nu in range(dim):
                    val = R_tensor[rho, sigma, mu, nu]
                    if val != 0:
                        val_latex = sp.latex(val)
                        eq_str = rf"R^{coord_names[rho]}_{{{coord_names[sigma]} \, {coord_names[mu]} \, {coord_names[nu]}}} = {val_latex}"
                        display(Math(eq_str))

def print_ricci_tensor(Ricci_tensor, coord_names):
    """Helper function to display only non-zero Ricci components using LaTeX."""
    dim = len(coord_names)
    for mu in range(dim):
        for nu in range(dim):
            val = Ricci_tensor[mu, nu]
            if val != 0:
                # Convert the SymPy expression to a LaTeX string
                val_latex = sp.latex(val)
                
                # Format the full equation
                eq_str = rf"R_{{{coord_names[mu]} \, {coord_names[nu]}}} = {val_latex}"
                
                # Render using IPython
                display(Math(eq_str))

def print_einstein_tensor(G, coord_names):
    """Helper function to display only non-zero Einstein components using LaTeX."""
    dim = len(coord_names)
    for mu in range(dim):
        for nu in range(dim):
            val = G[mu, nu]
            if val != 0:
                val_latex = sp.latex(val)
                
                # Format the full equation
                eq_str = rf"G_{{{coord_names[mu]} \, {coord_names[nu]}}} = {val_latex}"
                
                # Render using IPython
                display(Math(eq_str))

In [3]:
# Input your metric and coords here

print("Test: FLRW Metric")
t, r, theta, phi, rs, M, k = sp.symbols('t r theta phi rs M k') # all symbols
a = sp.Function('a')(t) # any functions

coords = [t, r, theta, phi] # coords
names = ['t', 'r', r'\theta', r'\phi'] # coords names

metric = sp.Matrix([
    [-1,          0,            0,       0],
    [0,           a**2/(1-k*r**2),         0,       0],
    [0,           0,            a**2 * r**2,    0],
    [0,           0,            0,       a**2 * r**2 * sp.sin(theta)**2]
])

Test: FLRW Metric


In [4]:
# Christoffel Symbols

gamma = compute_christoffel_universal(metric, coords)
print_non_zero_symbols(gamma, names)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
# Riemann Curvature Tensor

riemann = compute_riemann(gamma, coords)
print_riemann_symbols(riemann, names)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
# Ricci Tensor & Scalar

riccitensor = compute_ricci_tensor(riemann)
ricciscalar = compute_ricci_scalar(riccitensor, metric)
print_ricci_tensor(riccitensor, names)
display(Math(rf"Ricci \;Scalar = {sp.latex(ricciscalar)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [7]:
# Einstein Tensor

einstein = compute_einstein_tensor(riccitensor, ricciscalar, metric)
print_einstein_tensor(einstein, names)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>